# Preprocessing
This notebook serves to process the cleaned/merged data, addressing normalization, data sparsity, and class imbalance.

In [0]:
%pip install pandas
%pip install numpy

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np

In [0]:
# load the cleaned/merged parquet file from notebook 01:
input_path = '/Volumes/workspace/default/microbiome_project_files/processed/merged_data.parquet'
df = pd.read_parquet(input_path)

print(f"loaded {df.shape[0]} samples, {df.shape[1]} columns")
print(f"environment distribution:\n{df['empo_3'].value_counts()}")

loaded 432 samples, 31730 columns
environment distribution:
empo_3
Animal distal gut          71
Sediment (saline)          65
Plant surface              49
Soil (non-saline)          37
Water (saline)             36
Sediment (non-saline)      36
Animal corpus              33
Animal proximal gut        25
Subsurface (non-saline)    23
Water (non-saline)         22
Animal secretion           20
Fungus corpus              12
Surface (saline)            3
Name: count, dtype: int64


First, we'll filter out rare taxa as those are likely sequencing noise or contaminants. Removing them reduces dimensionality and will improve model generalization.

In [0]:
# filter out OTUs that only show up in less than 5% of the samples

# check shape beforehand
print(f"pre-filtered to {df.shape[0]} samples, {df.shape[1]} columns")

# separate out OTU columns from metadata columns
# these columns only have numbers in their ID
otu_cols = df.columns[df.columns.str.contains(r'^[0-9]+$')]

# count non-zero occurrences (not non-null)
prevalence = (df[otu_cols] > 0).sum(axis=0) / df.shape[0]

# get OTU columns that meet threshold
otus_to_keep = otu_cols[prevalence > 0.05]

# keep metadata columns AND filtered OTU columns
metadata_cols = [col for col in df.columns if col not in otu_cols]
df_filtered = df[metadata_cols + otus_to_keep.tolist()]

# check shape of df after filtering
print(f"filtered to {df_filtered.shape[0]} samples, {df_filtered.shape[1]} columns")
print("reduced column count by", df.shape[1] - df_filtered.shape[1], "columns")

pre-filtered to 432 samples, 31730 columns
filtered to 432 samples, 2219 columns
reduced column count by 29511 columns


Next, we'll apply the centered log-ratio (CLR) onto our OTU counts to normalize them, as that will remove bias from using raw counts (since microbiome data is compositional, with relative abundances).

In [0]:
# normalize counts using CLR (centered log-ratio) transformation

# pulled from: https://medium.com/@nextgendatascientist/a-guide-for-data-scientists-log-ratio-transformations-in-machine-learning-a2db44e2a455

def clr_transform(X):
    """
    Compute the Centered Logratio (CLR) transformation.
    """
    X = np.where(X == 0, 1e-6, X)  # replace zeroes with a psuedocount
    gm = np.exp(np.log(X).mean(axis=1, keepdims=True)) # calculate the geometric mean per sample
    return np.log(X / gm)

In [0]:
# apply the clr transform function

# extract out the OTU data only
otu_data = df_filtered[otus_to_keep].values

# apply the CLR transformation onto those values
otu_clr = clr_transform(otu_data)

# convert back to dataframe with same column names
otu_clr_df = pd.DataFrame(otu_clr, columns=otus_to_keep, index=df_filtered.index)

# recombine with metadata
df_normalized = pd.concat([df_filtered[metadata_cols], otu_clr_df], axis=1)

In [0]:
# export preprocessed data 
output_path = '/Volumes/workspace/default/microbiome_project_files/processed/normalized_data.parquet'
df_normalized.to_parquet(output_path)

In [0]:
# sanity checks
print(f"loaded {df_normalized.shape[0]} samples, {df_normalized.shape[1]} columns")
print(f"environment distribution:\n{df_normalized['empo_3'].value_counts()}")
print("reduced column count by", df.shape[1] - df_normalized.shape[1], "columns")

loaded 432 samples, 2219 columns
environment distribution:
empo_3
Animal distal gut          71
Sediment (saline)          65
Plant surface              49
Soil (non-saline)          37
Water (saline)             36
Sediment (non-saline)      36
Animal corpus              33
Animal proximal gut        25
Subsurface (non-saline)    23
Water (non-saline)         22
Animal secretion           20
Fungus corpus              12
Surface (saline)            3
Name: count, dtype: int64
reduced column count by 29511 columns
